# Multimodal Reranker with Qwen3-VL and OpenVINO

The Qwen3-VL-Reranker model series is built upon the powerful Qwen3-VL foundation model, designed to refine retrieval results by computing precise relevance scores for (query, document) pairs. Both query and document may contain arbitrary single or mixed modalities (text, images, screenshots, videos). In retrieval pipelines, the reranker is typically used in tandem with the embedding model: the embedding model performs efficient initial recall, while the reranker refines results in a subsequent re-ranking stage.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Embedding.png" width="500"/>

In this tutorial we consider how to convert and optimize Qwen3-VL Reranker model using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO model inference with Optimum-intel](#Run-OpenVINO-model-inference-with-Optimum-intel)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-reranker.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install "git+https://github.com/openvino-dev-samples/optimum-intel.git@qwen3vl-reranker" "transformers>=4.57.0" "torch>=2.9" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>=2025.4" "openvino_tokenizers>=2025.4"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [ ]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-reranker.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Reranker Model list:

| Model Type | Models | Size | Layers | Sequence Length | Instruction Aware |
|---|---|---|---|---|---|
| Multimodal Reranking | [Qwen3-VL-Reranker-2B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-2B) | 2B | 28 | 32K | Yes |
| Multimodal Reranking | [Qwen3-VL-Reranker-8B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-8B) | 8B | 36 | 32K | Yes |

In [ ]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Reranker-2B", "Qwen/Qwen3-VL-Reranker-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Reranker Model:",
)

model_selector

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [ ]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Reranker model can be exported by `image-text-to-text` task with Optimum-intel.

In [ ]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "image-text-to-text"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [ ]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

## Run OpenVINO model inference with Optimum-intel
[back to top ⬆️](#Table-of-contents:)

Select device from dropdown list for running inference using OpenVINO.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

The Qwen3-VL-Reranker model can be loaded by class `OVModelForVisualCausalLM` with Optimum-intel. Reranking works by computing yes/no token probabilities from the model's logits.

In [ ]:
from optimum.intel import OVModelForVisualCausalLM

model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value, export=False)

In [ ]:
import torch
from transformers import AutoProcessor


def format_reranker_inputs(processor, query, documents, instruction="Retrieve text relevant to the user's query."):
    """Format query-document pairs for the reranker model."""
    pairs = []
    for doc in documents:
        # Build content for the document
        doc_content = ""
        if isinstance(doc, str):
            doc_content = doc
        elif isinstance(doc, dict):
            doc_content = doc.get("text", "")

        query_text = query if isinstance(query, str) else query.get("text", "")

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": [{"type": "text", "text": f"<Query>: {query_text}\n<Document>: {doc_content}"}]},
        ]
        pairs.append(conversation)

    prompts = [
        processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
        for conv in pairs
    ]
    batch = processor(text=prompts, padding=True, return_tensors="pt")
    return batch


def compute_reranker_scores(model, tokenizer, inputs):
    """Compute relevance scores from yes/no token logits."""
    token_false_id = tokenizer.convert_tokens_to_ids("no")
    token_true_id = tokenizer.convert_tokens_to_ids("yes")

    outputs = model.forward(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

    logits = outputs.logits[:, -1, :]
    true_logits = logits[:, token_true_id]
    false_logits = logits[:, token_false_id]
    scores = torch.stack([false_logits, true_logits], dim=1)
    scores = torch.nn.functional.log_softmax(scores, dim=1)
    scores = scores[:, 1].exp().tolist()
    return scores


processor = AutoProcessor.from_pretrained(model_dir)
tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

# Define query and candidate documents
query = "What is the capital of China?"
documents = [
    "The capital of China is Beijing.",
    "Gravity is a fundamental force of nature.",
    "Python is a popular programming language.",
]

inputs = format_reranker_inputs(processor, query, documents)
scores = compute_reranker_scores(model, tokenizer, inputs)

# Print ranked results
print(f"Query: {query}\n")
for doc, score in sorted(zip(documents, scores), key=lambda x: x[1], reverse=True):
    print(f"  Score: {score:.4f} | {doc}")

In [ ]:
del model
import gc
gc.collect()